In [3]:
import sys

print("Python is working!")
print("Python version:", sys.version)


Python is working!
Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


In [4]:
import passwords as pw
import flickrapi
import pandas as pd
import pprint
import folium

api_key = pw.flickr_key
api_secret = pw.flickr_secret

# connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

In [5]:
# Define the location (latitude and longitude) and search parameters
centre_latitude = 45.437164
centre_longitude = 12.344833

# Define the bounding box. Use Open Street Map to get the coordinates
lat_north = 45.437313
lat_south = 45.436991
long_west = 12.344435
long_east = 12.345344

circle_radius = 2

# Format: bbox = "min_longitude,min_latitude,max_longitude,max_latitude" for Flickr search
bbox = str(long_west)+','+str(lat_south)+','+str(long_east)+','+str(lat_north)

# Define the number of photos to retrieve per page
per_page = 250

# Create a map centered on the location
map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=19,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False,
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=circle_radius,
    color="HotPink",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(circle_radius), # <--- CHANGE 'radius' to 'circle_radius'
).add_to(map_Venice)

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north, long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="HotPink",
).add_to(map_Venice)

map_Venice

In [6]:
    photos = flickr.photos.search(bbox=bbox, per_page=per_page, page=1, has_geo=1)

    total_pages = photos["photos"]["pages"]
    total_photos = photos["photos"]["total"]

    print(f"Total photos: {total_photos}")
    print(f"Total pages: {total_pages}")

Total photos: 62
Total pages: 1


In [7]:
print(bbox)


12.344435,45.436991,12.345344,45.437313


In [8]:
import pandas as pd
import numpy as np

# Define what you want your "Standard Notation" for missing data to be
MISSING_LABEL = "N/A"

def get_page(bbox, page, per_page):
    # 1. Add 'count_comments' to extras
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name,count_comments",
    )
    photos = response["photos"]["photo"]
    rows = []
    
    for photo in photos:
        # Default to MISSING_LABEL
        comments_text = MISSING_LABEL
        
        # Check if comments exist
        comment_count = int(photo.get('count_comments', 0))
        
        if comment_count > 0:
            try:
                comment_resp = flickr.photos.comments.getList(photo_id=photo['id'])
                if 'comments' in comment_resp and 'comment' in comment_resp['comments']:
                    comment_list = []
                    for c in comment_resp['comments']['comment']:
                        author = c.get('authorname', 'Unknown')
                        content = c.get('_content', '')
                        if content: # Only add if content is not empty
                            comment_list.append(f"{author}: {content}")
                    
                    if comment_list:
                        comments_text = " | ".join(comment_list)
            except Exception as e:
                print(f"Error fetching comments for {photo['id']}: {e}")

        # Safety Check: safely get description
        desc_content = photo.get("description", {}).get("_content", "")

        new_row = {
            "id": photo.get("id", MISSING_LABEL),
            "server": photo.get("server", MISSING_LABEL),
            "secret": photo.get("secret", MISSING_LABEL),
            "title": photo.get("title", ""), # Allow empty here, we catch it later
            "tags": photo.get("tags", ""),
            "views": photo.get("views", 0), # Keep numeric for views
            "description": desc_content,
            "date_taken": photo.get("datetaken", MISSING_LABEL),
            "latitude": photo.get("latitude", MISSING_LABEL),
            "longitude": photo.get("longitude", MISSING_LABEL),
            "url_s": photo.get("url_s", MISSING_LABEL),
            "owner": photo.get("owner", MISSING_LABEL),
            "owner_name": photo.get("ownername", MISSING_LABEL),
            "media": photo.get("media", MISSING_LABEL),
            "comments": comments_text,
        }
        rows.append(new_row)
        
    df = pd.DataFrame(
        rows,
        columns=[
            "id", "server", "secret", "title", "tags", "views", 
            "description", "date_taken", "latitude", "longitude", 
            "url_s", "owner_name", "owner", "media", "comments"
        ],
    )
    
    # ---------------------------------------------------------
    # FINAL CLEAN UP: Replace all empty types with Standard Notation
    # ---------------------------------------------------------
    
    # 1. Replace standard Pandas NaN (Not a Number) with "N/A"
    df = df.fillna(MISSING_LABEL)
    
    # 2. Replace empty strings "" with "N/A" (e.g., if title or tags are empty)
    df = df.replace(r'^\s*$', MISSING_LABEL, regex=True)
    
    return df

# Main Execution ----------------------------------------------
df = pd.DataFrame()

# Loop through pages
for page in range(1, 3):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}")
Output = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\Public_Spaces_Datascraping\Campo_San_Lorenzo\Campo_San_Lorenzo.csv"
df.to_csv(Output, index=False)

print("Done.")

Getting page 1 of 1
Total photos so far: 70
Getting page 2 of 1
Total photos so far: 70
Done.


In [ ]:
# Create a map centered on the location
map_Venice = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=19,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(circle_radius),    
).add_to(map_Venice) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=False,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",  
).add_to(map_Venice)

# Add a circle marker for each photo
latitudes = df['latitude']
longitudes = df['longitude']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid", 
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_Venice)
  
map_Venice